In [3]:
"""
Synthetic banking data generator.

Generates a dataset with the following columns:

Categorical
-----------
1.  PRDCT_TYPE_DSCR
2.  ACCT_PRD_TERM_MNTH_TXT
3.  segment_acct
4.  Acct_status
5.  New_cust_flag
6.  Acct_Open_Month (derived from date)
7.  bal_bucket
8.  channel
9.  fraud_flag
10. Freq

Continuous
----------
1. date
2. acct_open_cnt
3. balance
4. acct_close_cnt
"""

import numpy as np
import pandas as pd
from typing import Dict, List, Optional


# ---------------------------------------------------------------------------
# Helpers for numeric features
# ---------------------------------------------------------------------------

def _skewed_int_distribution(
    size: int,
    min_val: int,
    max_val: int,
    shape: float = 2.5,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """
    Generate integers in [min_val, max_val] with more mass near min_val using
    a Beta(shape, 1) distribution.

    This roughly mimics the heavy-tail you see in counts (many small values,
    few large ones).

    Parameters
    ----------
    size : int
        Number of samples to generate.
    min_val : int
        Minimum allowed value (inclusive).
    max_val : int
        Maximum allowed value (inclusive).
    shape : float, default 2.5
        Shape parameter for the Beta distribution. Larger => more skew
        toward min_val.
    rng : numpy.random.Generator, optional
        Random number generator instance for reproducibility.

    Returns
    -------
    numpy.ndarray
        Array of integers of length `size`.
    """
    if rng is None:
        rng = np.random.default_rng()

    if min_val > max_val:
        raise ValueError("min_val must be <= max_val")

    if min_val == max_val:
        return np.full(size, min_val, dtype=int)

    u = rng.beta(shape, 1.0, size=size)  # values in (0, 1)
    vals = min_val + (max_val - min_val) * u
    return np.floor(vals).astype(int)


def _skewed_float_distribution(
    size: int,
    min_val: float,
    max_val: float,
    shape: float = 2.5,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """
    Generate floats in [min_val, max_val] with more mass near min_val using
    a Beta(shape, 1) distribution.

    Good for balances: many small balances, fewer very large ones.

    Parameters
    ----------
    size : int
        Number of samples.
    min_val : float
        Minimum allowed value (inclusive).
    max_val : float
        Maximum allowed value (inclusive).
    shape : float, default 2.5
        Shape parameter for the Beta distribution.
    rng : numpy.random.Generator, optional
        Random number generator instance.

    Returns
    -------
    numpy.ndarray
        Array of floats of length `size`.
    """
    if rng is None:
        rng = np.random.default_rng()

    if min_val > max_val:
        raise ValueError("min_val must be <= max_val")

    if np.isclose(min_val, max_val):
        return np.full(size, float(min_val))

    u = rng.beta(shape, 1.0, size=size)
    vals = min_val + (max_val - min_val) * u
    return vals.astype(float)


# ---------------------------------------------------------------------------
# Helpers for categorical features
# ---------------------------------------------------------------------------

def _sample_categorical(
    size: int,
    categories: List,
    probs: Optional[List[float]] = None,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """
    Sample categorical values.

    Parameters
    ----------
    size : int
        Number of samples.
    categories : list
        Category labels.
    probs : list of float, optional
        Probabilities for each category. If None, uses uniform.
    rng : numpy.random.Generator, optional
        Random generator instance.

    Returns
    -------
    numpy.ndarray
        Array of sampled category labels.
    """
    if rng is None:
        rng = np.random.default_rng()

    categories = np.array(categories, dtype=object)
    if probs is None:
        probs = np.full(len(categories), 1.0 / len(categories))
    else:
        probs = np.array(probs, dtype=float)
        if not np.isclose(probs.sum(), 1.0):
            probs = probs / probs.sum()

    return rng.choice(categories, size=size, p=probs)


def _default_category_config() -> Dict[str, Dict[str, List]]:
    """
    Default configuration for categorical columns, inferred from your
    screenshots. You can update the values/probs to better match your
    real data if you compute empirical frequencies.

    Returns
    -------
    dict
        Mapping of column name -> {'values': [...], 'probs': [...]}
    """
    return {
        "PRDCT_TYPE_DSCR": {
            "values": [
                "Basic Checking",
                "Dream Account",
                "Online Certificate of Deposit",
                "Online Savings",
                "Select CD",
                "Select Savings",
                "Tiered Savings",
            ],
            # Heavier weight on Basic Checking
            "probs": [0.55, 0.15, 0.05, 0.08, 0.05, 0.07, 0.05],
        },
        "ACCT_PRD_TERM_MNTH_TXT": {
            "values": [
                "11M", "12M", "15M", "18M", "24M", "30M",
                "36M", "3M", "48M", "60M", "6M", "7M", "8M", "9M"
            ],
            "probs": None,  # roughly uniform
        },
        "segment_acct": {
            "values": [
                "1. backbook acct - MoB>=3",
                "0. closed acct",
                "2. new acct - MoB<3",
            ],
            "probs": [0.7, 0.15, 0.15],
        },
        "Acct_status": {
            "values": ["Open_funded", "Closed", "Open_Not_funded"],
            "probs": [0.75, 0.15, 0.10],
        },
        "New_cust_flag": {
            "values": [0, 1],
            "probs": [0.8, 0.2],
        },
        "bal_bucket": {
            "values": [
                "1. <$1K",
                "2. $1K-$10K",
                "3. $10K-$25K",
                "4. $25K-$50K",
                "5. $50K-$100K",
                "6. $100K-$250K",
                "7. >$250K",
            ],
            "probs": [0.25, 0.25, 0.15, 0.12, 0.10, 0.08, 0.05],
        },
        "channel": {
            "values": [
                "Other",
                "Affiliates",
                "Existing Customers",
                "Paid Display",
                "Paid Search",
                "Paid Social",
                "Barclays Storefront",
                "Deposits Storefront",
            ],
            "probs": [0.5, 0.08, 0.06, 0.06, 0.10, 0.10, 0.05, 0.05],
        },
        "fraud_flag": {
            "values": [0, 1],
            "probs": [0.99, 0.01],
        },
        "Freq": {
            "values": ["Daily", "Weekly", "MTD", "YTD", "Monthly"],
            "probs": [0.7, 0.1, 0.05, 0.05, 0.10],
        },
    }


# ---------------------------------------------------------------------------
# Date & month generation
# ---------------------------------------------------------------------------

def _generate_dates(
    size: int,
    min_date: str,
    max_date: str,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    """
    Generate random dates uniformly between min_date and max_date.

    Parameters
    ----------
    size : int
        Number of samples.
    min_date : str or datetime-like
        Lower bound for dates (inclusive).
    max_date : str or datetime-like
        Upper bound for dates (inclusive).
    rng : numpy.random.Generator, optional
        Random generator instance.

    Returns
    -------
    numpy.ndarray
        Array of pandas.Timestamp objects.
    """
    if rng is None:
        rng = np.random.default_rng()

    min_ts = pd.to_datetime(min_date)
    max_ts = pd.to_datetime(max_date)

    if min_ts > max_ts:
        raise ValueError("min_date must be <= max_date")

    delta_days = (max_ts - min_ts).days
    offsets = rng.integers(0, delta_days + 1, size=size)
    dates = min_ts + pd.to_timedelta(offsets, unit="D")
    return dates.values


def _derive_acct_open_month_from_date(dates: np.ndarray) -> np.ndarray:
    """
    Derive the `Acct_Open_Month` label from actual dates.

    * If date < 2020-01-01  -> 'Prior 2020'
    * Else                  -> 'YYYYMM' (e.g. 202505)

    Parameters
    ----------
    dates : numpy.ndarray
        Array of datetime64 values.

    Returns
    -------
    numpy.ndarray
        Array of strings representing account open month.
    """
    dates_ts = pd.to_datetime(dates)
    cutoff = pd.Timestamp("2020-01-01")

    labels = []
    for dt in dates_ts:
        if dt < cutoff:
            labels.append("Prior 2020")
        else:
            labels.append(f"{dt.year}{dt.month:02d}")
    return np.array(labels, dtype=object)


# ---------------------------------------------------------------------------
# Main public API
# ---------------------------------------------------------------------------

def generate_synthetic_banking_data(
    n_rows: int,
    min_date: str,
    max_date: str,
    min_acct_open_cnt: int,
    max_acct_open_cnt: int,
    min_balance: float,
    max_balance: float,
    min_acct_close_cnt: int,
    max_acct_close_cnt: int,
    seed: Optional[int] = None,
    category_config: Optional[Dict[str, Dict[str, List]]] = None,
) -> pd.DataFrame:
    """
    Generate a synthetic banking dataset that mimics the structure and rough
    distributions of the provided sample.

    Parameters
    ----------
    n_rows : int
        Number of data points (rows) to generate.
    min_date, max_date : str or datetime-like
        Bounds for the `date` column.
    min_acct_open_cnt, max_acct_open_cnt : int
        Bounds for `acct_open_cnt`.
    min_balance, max_balance : float
        Bounds for `balance`.
    min_acct_close_cnt, max_acct_close_cnt : int
        Bounds for `acct_close_cnt`.
    seed : int, optional
        Random seed for reproducibility.
    category_config : dict, optional
        Custom categorical configuration; same format as
        `_default_category_config()`. If None, defaults are used.

    Returns
    -------
    pandas.DataFrame
        Synthetic dataset with all requested columns.
    """
    rng = np.random.default_rng(seed)

    if category_config is None:
        category_config = _default_category_config()

    # ---- Continuous columns ----
    dates = _generate_dates(n_rows, min_date, max_date, rng=rng)

    acct_open_cnt = _skewed_int_distribution(
        n_rows, min_acct_open_cnt, max_acct_open_cnt, shape=2.5, rng=rng
    )
    balance = _skewed_float_distribution(
        n_rows, min_balance, max_balance, shape=2.0, rng=rng
    )
    acct_close_cnt = _skewed_int_distribution(
        n_rows, min_acct_close_cnt, max_acct_close_cnt, shape=2.8, rng=rng
    )

    # ---- Categorical columns ----
    cat_cols = {}
    for col, cfg in category_config.items():
        cat_cols[col] = _sample_categorical(
            n_rows, cfg["values"], cfg.get("probs"), rng=rng
        )

    # Derive open month from actual date
    acct_open_month = _derive_acct_open_month_from_date(dates)

    # Assemble final DataFrame in your desired column order
    df = pd.DataFrame(
        {
            "PRDCT_TYPE_DSCR": cat_cols["PRDCT_TYPE_DSCR"],
            "ACCT_PRD_TERM_MNTH_TXT": cat_cols["ACCT_PRD_TERM_MNTH_TXT"],
            "segment_acct": cat_cols["segment_acct"],
            "Acct_status": cat_cols["Acct_status"],
            "New_cust_flag": cat_cols["New_cust_flag"],
            "Acct_Open_Month": acct_open_month,
            "bal_bucket": cat_cols["bal_bucket"],
            "channel": cat_cols["channel"],
            "fraud_flag": cat_cols["fraud_flag"],
            "Freq": cat_cols["Freq"],
            "date": dates,
            "acct_open_cnt": acct_open_cnt,
            "balance": balance,
            "acct_close_cnt": acct_close_cnt,
        }
    )

    return df


def save_synthetic_banking_data_to_excel(
    df: pd.DataFrame,
    filepath: str,
    sheet_name: str = "synthetic_data",
) -> None:
    """
    Save the synthetic DataFrame to an Excel file.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame produced by `generate_synthetic_banking_data`.
    filepath : str
        Path of the Excel file to create, e.g. 'synthetic_banking_data.xlsx'.
    sheet_name : str, default 'synthetic_data'
        Sheet name for the output workbook.
    """
    df.to_excel(filepath, sheet_name=sheet_name, index=False)




In [4]:
# ---------------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------------

# You can adjust all of these parameters as needed
example_df = generate_synthetic_banking_data(
    n_rows=1_00_000,
    min_date="2024-01-01",
    max_date="2025-05-31",
    min_acct_open_cnt=0,
    max_acct_open_cnt=60000,
    min_balance=0.0,
    max_balance=12_000_000_000.0,
    min_acct_close_cnt=0,
    max_acct_close_cnt=166893,
    seed=42,
)

save_synthetic_banking_data_to_excel(
    example_df,
    filepath="synthetic_banking_data.xlsx",
)
